# 03 - Entraînement du modèle

On recharge les données brutes et on applique la normalisation sauvegardée dans `normalization.json` (calculée dans `02_preprocessing.ipynb`), sans la recalculer : sinon les valeurs seraient légèrement différentes de celles utilisées plus tard dans la simulation.

Modèle choisi : un petit CNN 1D. Il capte les motifs répétés dans le temps, a peu de paramètres, et dispose d'un chemin de quantification int8 simple et mature dans TensorFlow Lite. Un LSTM est techniquement supporté par TensorFlow Lite Micro mais avec des contraintes fortes (variante standard uniquement, format de quantification particulier pour l'état interne).

In [ ]:
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

DATA_DIR = Path("../data/raw/UCI HAR Dataset")
MODELS_DIR = Path("../models")
RESULTS_DIR = Path("../results")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Recharger les données et la normalisation figée à l'étape précédente

In [ ]:
with open(MODELS_DIR / "normalization.json") as f:
    normalization = json.load(f)

CHANNELS = normalization["channels"]
mean = np.array(normalization["mean"])
std = np.array(normalization["std"])
validation_subjects = normalization["validation_subjects"]

def load_ids(path):
    return pd.read_csv(path, header=None, names=["value"]).squeeze("columns")

def load_signal(split, signal_name):
    path = DATA_DIR / split / "Inertial Signals" / f"{signal_name}_{split}.txt"
    return np.loadtxt(path)

def load_windows(split):
    signals = [load_signal(split, name) for name in CHANNELS]
    return np.stack(signals, axis=-1)

def normalize(X):
    return (X - mean) / std

X_train_full = load_windows("train")
X_test = normalize(load_windows("test"))
subject_train_full = load_ids(DATA_DIR / "train" / "subject_train.txt")
y_train_full = (load_ids(DATA_DIR / "train" / "y_train.txt") - 1).to_numpy()
y_test = (load_ids(DATA_DIR / "test" / "y_test.txt") - 1).to_numpy()

val_mask = subject_train_full.isin(validation_subjects).to_numpy()
fit_mask = ~val_mask

X_fit = normalize(X_train_full[fit_mask])
y_fit = y_train_full[fit_mask]
X_val = normalize(X_train_full[val_mask])
y_val = y_train_full[val_mask]

print("X_fit:", X_fit.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)

## Architecture du CNN 1D

- Deux couches `Conv1D` extraient des motifs locaux, une couche de pooling réduit la longueur de séquence.
- `GlobalAveragePooling1D` plutôt qu'un `Flatten` : résume chaque canal en une valeur, donc beaucoup moins de paramètres dans la couche dense suivante.
- `Dropout` avant la dernière couche pour limiter le sur-apprentissage, vu la taille modeste du jeu d'entraînement (5551 fenêtres).

In [ ]:
n_classes = len(np.unique(y_fit))

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(128, len(CHANNELS))),
    tf.keras.layers.Conv1D(16, kernel_size=5, padding="same", activation="relu"),
    tf.keras.layers.Conv1D(32, kernel_size=5, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.Conv1D(32, kernel_size=3, padding="same", activation="relu"),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(n_classes, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## Entraînement

`EarlyStopping` sur la perte de validation : si elle ne s'améliore plus pendant 15 époques, on arrête et on garde les meilleurs poids rencontrés. Ça évite de fixer arbitrairement un nombre d'époques et limite le sur-apprentissage.

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=15, restore_best_weights=True
)

history = model.fit(
    X_fit, y_fit,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=2,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="validation")
axes[1].set_title("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_curves.png")
plt.show()

## Sauvegarde du modèle entraîné

Format Keras natif (`.keras`) pour l'instant. La conversion en TensorFlow Lite se fait dans le notebook suivant, une fois le modèle évalué et figé.

In [ ]:
model.save(MODELS_DIR / "model.keras")

val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=0)
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"validation accuracy: {val_accuracy:.3f}")
print(f"test accuracy (indicative only, real evaluation in the next notebook): {test_accuracy:.3f}")

## Observations

_À compléter après exécution : nombre de paramètres, nombre d'époques réellement effectuées avant l'arrêt anticipé, allure des courbes, accuracy obtenue._